In [1]:
from pathlib import Path
import geopandas as gpd
from shapely.geometry import LineString, Polygon, box
import rasterio
import os
import matplotlib.pyplot as plt
import folium
from branca.colormap import LinearColormap
import os
import numpy as np
from shapely.ops import transform
import pyproj

In [2]:
region_list = ["ARK-NZK","Vallei en Veluwe",
               "Achterhoek", "Brabantse Delta","Friesland",
               "Groningen en NO-Drenthe","Limburg",
               "Noord-Brabant Oost","Noord-Westelijke Delta",
               "Rivierenland","Scheldestromen","Zuiderzeeland",
               "Overijsselse Vecht"
               ]

In [ ]:
region_list = ["Achterhoek"]

In [17]:
region_list = [
               "ARK-NZK", "Brabantse Delta","Noord-Brabant Oost"
               ]

In [18]:
# Assume lower_lying_regions contains both line and polygon layers

from pathlib import Path
import geopandas as gpd
import pandas as pd

lowerlying_dir = Path(r"P:\bovenregionale-stresstest-hwn\Data\Lowerlying_sections")
lowerlying_gpkgs = list(lowerlying_dir.glob("*.gpkg"))
print(f"Found {len(lowerlying_gpkgs)} lowerlying GeoPackages.")

# Read all GeoPackages into a list of GeoDataFrames
lower_lying_regions = [gpd.read_file(gpkg) for gpkg in lowerlying_gpkgs]

buffer_distance = 1  # meters

buffered_gdfs = []
polygon_gdfs = []

for gdf in lower_lying_regions:
    geom_types = set(gdf.geometry.geom_type)
    if 'LineString' in geom_types or 'MultiLineString' in geom_types:
        # Buffer lines to polygons
        gdf_buffered = gdf.copy()
        gdf_buffered['geometry'] = gdf_buffered.geometry.buffer(buffer_distance)
        buffered_gdfs.append(gdf_buffered)
    else:
        polygon_gdfs.append(gdf)

# Combine all as polygons
all_polygons = buffered_gdfs + polygon_gdfs

# Ensure all have the same CRS
target_crs = all_polygons[0].crs
all_polygons = [gdf.to_crs(target_crs) for gdf in all_polygons]

# Merge into one GeoDataFrame
merged_lower_lying = gpd.GeoDataFrame(
    pd.concat(all_polygons, ignore_index=True),
    crs=target_crs
)

output_path = lowerlying_dir / "merged_lower_lying_buffered.gpkg"
if output_path.exists():
    output_path.unlink()
merged_lower_lying.to_file(output_path, driver="GPKG")
print(f"Merged (buffered) GeoDataFrame written to: {output_path}")

Found 5 lowerlying GeoPackages.


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'P:\\bovenregionale-stresstest-hwn\\Data\\Lowerlying_sections\\merged_lower_lying_buffered.gpkg'

In [19]:
from post_processing_functions import Thresholding_for_artefacts,Thresholding_for_artefacts,Filter_and_aggregate_flooded_segments_exposure, Filter_and_aggregate_flooded_segments_damage, calculate_overlay_percentages,get_z_height_optimized
import pandas as pd 
# add tunnels and bridge % columns to exposure and damage files and filter

data_dir = Path(r"P:\bovenregionale-stresstest-hwn\Data\Processed_data")


tunnels = data_dir.joinpath("Tunnels_filtered_by_area_2000_th.gpkg")
bridges = data_dir.joinpath("filtered_bridges.gpkg")
kunstinweg = data_dir.joinpath("kunstinweg.shp")
road_height_path = data_dir.joinpath("road_height_points.gpkg")

kunstinweg_gdf = gpd.read_file(kunstinweg)
kunstinweg_gdf['geometry'] = kunstinweg_gdf['geometry'].buffer(0.2) # to make sure lines are valid
kunstinweg_gdf.rename(columns={'OMSCHR': 'objecttekst'}, inplace=True)

kunstinweg_bridge = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'brug']
kunstinweg_tunnel = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'tunnel']


tunnels_gdf_kunstoverweg = gpd.read_file(tunnels)
bridges_gdf_kunstoverweg = gpd.read_file(bridges) 

bridges_gdf = gpd.GeoDataFrame(
    pd.concat([bridges_gdf_kunstoverweg, kunstinweg_bridge], ignore_index=True),
    crs=bridges_gdf_kunstoverweg.crs
)

tunnels_gdf = gpd.GeoDataFrame(
    pd.concat([tunnels_gdf_kunstoverweg, kunstinweg_tunnel], ignore_index=True),
    crs=tunnels_gdf_kunstoverweg.crs
)



# Define allowed values for bridges and tunnels (lowercased for case-insensitive matching)
allowed_bridges = [
    'aanbrug', 'brug', 'brug (beweegbaar)', 'brug (landbouw)', 'brug (vast)',
    'brug beton', 'brug beton in', 'brug beton over', 'brug beweegbaar',
    'brug hout in', 'brug in', 'brug in de toerit va', 'brug staal in',
    'brug vast', 'vaste brug'
]

allowed_tunnels = [
    'cervedict tunnel', 'open tunnelbak', 'tunnel', 'tunnel vlak',
    'tunnelbak', 'tunnelbak den kaat'
]

filtered_viaducts = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'viaduct']

# Convert to lowercase for case-insensitive comparison
allowed_bridges = [x.lower() for x in allowed_bridges]
allowed_tunnels = [x.lower() for x in allowed_tunnels]

# Filter bridges
filtered_gdf_brug = bridges_gdf[
    bridges_gdf['objecttekst'].str.lower().isin(allowed_bridges)
]

# Filter tunnels
filtered_gdf_tunnel_and_bridges = tunnels_gdf[
    tunnels_gdf['objecttekst'].str.lower().isin(allowed_tunnels + allowed_bridges)
]


In [20]:
from post_processing_functions import Thresholding_for_artefacts,Aggregate_flooded_segments, calculate_overlay_percentages,get_z_height_optimized


for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "damages/HZ_damage_segmented.gpkg"
    
    roads_ex_gdf = gpd.read_file(roads_ex)
    
    points_gdf = gpd.read_file(road_height_path)
    points_gdf = points_gdf.to_crs(roads_ex_gdf.crs)
    roads_ex_gdf['Z_height'] = get_z_height_optimized(roads_ex_gdf, points_gdf, threshold=50.0)
    
    print("Calculating tunnel and bridge percentages...")
    Roads_assets = calculate_overlay_percentages(roads_ex_gdf, filtered_gdf_brug, filtered_gdf_tunnel_and_bridges,filtered_viaducts,merged_lower_lying)
    print("Applying thresholding to remove artefacts...")
    dataframe = Thresholding_for_artefacts("F_","Damages", Roads_assets, root_dir)
    print(dataframe.columns)
    print("Filtering and aggregating flooded segments...")
    Aggregate_flooded_segments(dataframe, root_dir,"Aggregated", dissolve_col='NETWERKSCH_HWN')
    


Processing region: ARK-NZK network
Calculating tunnel and bridge percentages...
Applying thresholding to remove artefacts...


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Damages_Artefact_Bridges_Viaducts_Tunnels_removed')) failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Damages_Artefact_Bridges_Viaducts_Tunnels_removed')) failed: disk I/O error"


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Damages_Filtering_all_columns')) failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Damages_Filtering_all_columns')) failed: disk I/O error"


...

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 836 WHERE lower(table_name) = lower('Damages_Artefact_Bridges_Viaducts_Tunnels_removed')) failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 836 WHERE lower(table_name) = lower('Damages_Artefact_Bridges_Viaducts_Tunnels_removed')) failed: disk I/O error"


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET min_x = 116717.9596000019, min_y = 347951.349300001, max_x = 206578.1863999999, max_y = 426288.1977999989 WHERE lower(table_name) = lower('Damages_Filtering_all_columns') AND Lower(data_type) = 'features') failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET min_x = 116717.9596000019, min_y = 347951.349300001, max_x = 206578.1863999999, max_y = 426288.1977999989 WHERE lower(table_name) = lower('Damages_Filtering_all_columns') AND Lower(data_type) = 'features') failed: disk I/O error"


Index(['link_id', 'REF_ID', 'infra_type', 'avgspeed', 'lanes', 'id_NWB',
       'BST_CODE_NWB', 'WVK_ID', 'FOW_NWB', 'HECTO_LTTR_NWB', 'NETWERKSCH_HWN',
       'NWBWGNR_HWN', 'NWBWGDL_HWN', 'node_A', 'node_B', 'edge_fid', 'rfid_c',
       'rfid', 'length', 'time', 'F_EV1_mi', 'F_EV1_ma', 'F_EV1_me',
       'F_EV1_fr', 'F_EV2_mi', 'F_EV2_ma', 'F_EV2_me', 'F_EV2_fr', 'road_type',
       'lanes_copy', 'dam_EV1_HZ', 'dam_EV2_HZ', 'geometry', 'Z_height',
       'tunnel_percentage', 'bridge_percentage', 'viaduct_percentage',
       'lowerlying_percentage', 'OR_me_10_fr_25', 'AND_me_10_fr_20', 'culvert',
       'me_20', 'isolated', 'is_flooded', 'Asset', 'flooded_bridge',
       'artefact', 'remove', 'is_isolated', 'flooded_tunnel_entrance',
       'flooded_lowerlying_entrance'],
      dtype='object')
Filtering and aggregating flooded segments...


In [21]:
#Flooded tunnel entrances


for region in region_list:
    print(f"Processing region: {region} tunnels")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "Damages_Filtering_all_columns.gpkg"
    roads_ex_gdf = gpd.read_file(roads_ex)

    # Use only allowed tunnels (not bridges)
    allowed_tunnels_only = tunnels_gdf[
        tunnels_gdf['objecttekst'].str.lower().isin(allowed_tunnels)
    ].copy()

    # Drop invalid/empty geometries
    allowed_tunnels_only = allowed_tunnels_only[
        allowed_tunnels_only.geometry.notna() & ~allowed_tunnels_only.geometry.is_empty
    ].copy()

    # Ensure CRS match
    if allowed_tunnels_only.crs != roads_ex_gdf.crs:
        allowed_tunnels_only = allowed_tunnels_only.to_crs(roads_ex_gdf.crs)

    # Keep only the flag and geometry from roads, coerce to 0/1
    if 'flooded_tunnel_entrance' not in roads_ex_gdf.columns:
        raise KeyError("Column 'flooded_tunnel_entrance' not found in roads_ex_gdf")
    roads_flags = roads_ex_gdf[['flooded_tunnel_entrance', 'geometry']].copy()
    roads_flags['flooded_tunnel_entrance'] = (
        roads_flags['flooded_tunnel_entrance'].fillna(0).astype(float).gt(0).astype(int)
    )

    # Spatial join: tunnels vs roads
    joined = gpd.sjoin(
        allowed_tunnels_only,
        roads_flags,
        how='left',
        predicate='intersects'
    )

    # Aggregate per tunnel: flooded if any intersecting road has flag == 1
    flooded_by_tunnel = (
        joined.groupby(joined.index)['flooded_tunnel_entrance']
        .max()
        .reindex(allowed_tunnels_only.index)
        .fillna(0)
        .astype(int)
    )

    # Add the flooded flag (per tunnel)
    tunnels_out = allowed_tunnels_only.copy()
    tunnels_out['flooded'] = flooded_by_tunnel.values

    # Save per-tunnel result
    output_gpkg = root_dir.joinpath("allowed_tunnels_flooded.gpkg")
    if output_gpkg.exists():
        output_gpkg.unlink()
    tunnels_out.to_file(output_gpkg, driver="GPKG")
    print(f"Saved: {output_gpkg} | flooded={(tunnels_out['flooded']==1).sum()} / {len(tunnels_out)}")

    # ---- Dissolve tunnels that touch OR are within 1 m ----
    TOLERANCE_M = 1.0  # meters

    # Clean index for spatial index bookkeeping
    tunnels_out = tunnels_out.reset_index(drop=True)

    # Skip if empty
    if len(tunnels_out) == 0 or tunnels_out.geometry.is_empty.all():
        print("No tunnel features to dissolve.")
        continue

    # Build proximity connectivity using buffered queries (distance <= TOLERANCE_M)
    try:
        buffered = tunnels_out.geometry.buffer(TOLERANCE_M)
        idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')
    except Exception as e:
        print(f"Spatial index query failed: {e}")
        idx_src = idx_tgt = []

    # Union-Find for connected components
    n = len(tunnels_out)
    parent = list(range(n))
    rank = [0] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    # Add edges (skip self-pairs)
    for a, b in zip(idx_src, idx_tgt):
        if a != b:
            if b < a:
                a, b = b, a
            union(int(a), int(b))

    # Component ids
    roots = [find(i) for i in range(n)]
    unique_roots = {r: i for i, r in enumerate(sorted(set(roots)))}
    tunnels_out['comp_id'] = [unique_roots[r] for r in roots]

    # Aggregate attributes: flooded=max; others=first
    agg = {c: 'first' for c in tunnels_out.columns if c not in ['geometry', 'flooded', 'comp_id']}
    agg['flooded'] = 'max'

    tunnels_out_diss = tunnels_out.dissolve(by='comp_id', aggfunc=agg).reset_index(drop=True)

    # Save dissolved result
    output_gpkg_diss = root_dir.joinpath("allowed_tunnels_flooded_dissolved.gpkg")
    if output_gpkg_diss.exists():
        output_gpkg_diss.unlink()
    tunnels_out_diss.to_file(output_gpkg_diss, driver="GPKG")
    print(
        f"Saved: {output_gpkg_diss} | groups={len(tunnels_out_diss)} | "
        f"flooded={(tunnels_out_diss['flooded']==1).sum()}"
    )


Processing region: ARK-NZK tunnels
Saved: P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Outputs\allowed_tunnels_flooded.gpkg | flooded=16 / 266


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_16168\3867464591.py:74: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('allowed_tunnels_flooded_dissolved')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('allowed_tunnels_flooded_dissolved')) failed: unable to open database file"


Saved: P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Outputs\allowed_tunnels_flooded_dissolved.gpkg | groups=58 | flooded=10
Processing region: Brabantse Delta tunnels
Saved: P:\bovenregionale-stresstest-hwn\Analysis\Brabantse Delta\Outputs\allowed_tunnels_flooded.gpkg | flooded=0 / 266


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_16168\3867464591.py:74: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')


Saved: P:\bovenregionale-stresstest-hwn\Analysis\Brabantse Delta\Outputs\allowed_tunnels_flooded_dissolved.gpkg | groups=58 | flooded=0
Processing region: Noord-Brabant Oost tunnels
Saved: P:\bovenregionale-stresstest-hwn\Analysis\Noord-Brabant Oost\Outputs\allowed_tunnels_flooded.gpkg | flooded=0 / 266


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_16168\3867464591.py:74: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')


Saved: P:\bovenregionale-stresstest-hwn\Analysis\Noord-Brabant Oost\Outputs\allowed_tunnels_flooded_dissolved.gpkg | groups=58 | flooded=0


In [22]:
for region in region_list:
    print(f"Processing region: {region} lowerlying entrances")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "Damages_Filtering_all_columns.gpkg"
    roads_ex_gdf = gpd.read_file(roads_ex)

    # Use only segments with partial lowerlying percentage
    allowed_lowerlying_only = roads_ex_gdf[
        (roads_ex_gdf['lowerlying_percentage'] > 0) & (roads_ex_gdf['lowerlying_percentage'] <= 100)
    ].copy()

    # Drop invalid/empty geometries
    allowed_lowerlying_only = allowed_lowerlying_only[
        allowed_lowerlying_only.geometry.notna() & ~allowed_lowerlying_only.geometry.is_empty
    ].copy()

    # Remove the column from left DataFrame to avoid suffixes
    if 'flooded_lowerlying_entrance' in allowed_lowerlying_only.columns:
        allowed_lowerlying_only = allowed_lowerlying_only.drop(columns=['flooded_lowerlying_entrance'])

    # Ensure CRS match (should already match)
    # Keep only the flag and geometry from roads, coerce to 0/1
    if 'flooded_lowerlying_entrance' not in roads_ex_gdf.columns:
        raise KeyError("Column 'flooded_lowerlying_entrance' not found in roads_ex_gdf")
    roads_flags = roads_ex_gdf[['flooded_lowerlying_entrance', 'geometry']].copy()
    roads_flags['flooded_lowerlying_entrance'] = (
        roads_flags['flooded_lowerlying_entrance'].fillna(0).astype(float).gt(0).astype(int)
    )

    # Spatial join: lowerlying vs roads
    joined = gpd.sjoin(
        allowed_lowerlying_only,
        roads_flags,
        how='left',
        predicate='intersects'
    )

    # Aggregate per segment: flooded if any intersecting road has flag == 1
    flooded_by_lowerlying = (
        joined.groupby(joined.index)['flooded_lowerlying_entrance']
        .max()
        .reindex(allowed_lowerlying_only.index)
        .fillna(0)
        .astype(int)
    )

    # Add the flooded flag (per segment)
    lowerlying_out = allowed_lowerlying_only.copy()
    lowerlying_out['flooded'] = flooded_by_lowerlying.values

    # Save per-segment result
    output_gpkg = root_dir.joinpath("allowed_lowerlying_flooded.gpkg")
    if output_gpkg.exists():
        output_gpkg.unlink()
    lowerlying_out.to_file(output_gpkg, driver="GPKG")
    print(f"Saved: {output_gpkg} | flooded={(lowerlying_out['flooded']==1).sum()} / {len(lowerlying_out)}")

    # ---- Dissolve segments that touch OR are within 1 m ----
    TOLERANCE_M = 1.0  # meters

    # Clean index for spatial index bookkeeping
    lowerlying_out = lowerlying_out.reset_index(drop=True)

    # Skip if empty
    if len(lowerlying_out) == 0 or lowerlying_out.geometry.is_empty.all():
        print("No lowerlying features to dissolve.")
        continue

    # Build proximity connectivity using buffered queries (distance <= TOLERANCE_M)
    try:
        buffered = lowerlying_out.geometry.buffer(TOLERANCE_M)
        idx_src, idx_tgt = lowerlying_out.sindex.query_bulk(buffered, predicate='intersects')
    except Exception as e:
        print(f"Spatial index query failed: {e}")
        idx_src = idx_tgt = []

    # Union-Find for connected components
    n = len(lowerlying_out)
    parent = list(range(n))
    rank = [0] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    # Add edges (skip self-pairs)
    for a, b in zip(idx_src, idx_tgt):
        if a != b:
            if b < a:
                a, b = b, a
            union(int(a), int(b))

    # Component ids
    roots = [find(i) for i in range(n)]
    unique_roots = {r: i for i, r in enumerate(sorted(set(roots)))}
    lowerlying_out['comp_id'] = [unique_roots[r] for r in roots]

    # Aggregate attributes: flooded=max; others=first
    agg = {c: 'first' for c in lowerlying_out.columns if c not in ['geometry', 'flooded', 'comp_id']}
    agg['flooded'] = 'max'

    lowerlying_out_diss = lowerlying_out.dissolve(by='comp_id', aggfunc=agg).reset_index(drop=True)

    # Save dissolved result
    output_gpkg_diss = root_dir.joinpath("allowed_lowerlying_flooded_dissolved.gpkg")
    if output_gpkg_diss.exists():
        output_gpkg_diss.unlink()
    lowerlying_out_diss.to_file(output_gpkg_diss, driver="GPKG")
    print(
        f"Saved: {output_gpkg_diss} | groups={len(lowerlying_out_diss)} | "
        f"flooded={(lowerlying_out_diss['flooded']==1).sum()}"
    )

Processing region: ARK-NZK lowerlying entrances


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 121 WHERE lower(table_name) = lower('allowed_lowerlying_flooded')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_ogr_contents SET feature_count = 121 WHERE lower(table_name) = lower('allowed_lowerlying_flooded')) failed: unable to open database file"
C:\Users\gunaratn\AppData\Local\Temp\ipykernel_16168\4021925880.py:72: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = lowerlying_out.sindex.query_bulk(buffered, predicate='intersects')


Saved: P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Outputs\allowed_lowerlying_flooded.gpkg | flooded=43 / 121


CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('allowed_lowerlying_flooded_dissolved')) failed: unable to open database file"

...

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('allowed_lowerlying_flooded')) failed: unable to open database file"
C:\Users\gunaratn\AppData\Local\Temp\ipykernel_16168\4021925880.py:72: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = lowerlying_out.sindex.query_bulk(buffered, predicate='intersects')


Saved: P:\bovenregionale-stresstest-hwn\Analysis\Noord-Brabant Oost\Outputs\allowed_lowerlying_flooded.gpkg | flooded=20 / 57


CPLE_AppDefinedError: b'sqlite3_exec(CREATE TABLE gpkg_extensions (table_name TEXT,column_name TEXT,extension_name TEXT NOT NULL,definition TEXT NOT NULL,scope TEXT NOT NULL,CONSTRAINT ge_tce UNIQUE (table_name, column_name, extension_name))) failed: disk I/O error'

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b'sqlite3_exec(CREATE TABLE gpkg_extensions (table_name TEXT,column_name TEXT,extension_name TEXT NOT NULL,definition TEXT NOT NULL,scope TEXT NOT NULL,CONSTRAINT ge_tce UNIQUE (table_name, column_name, extension_name))) failed: disk I/O error'


Saved: P:\bovenregionale-stresstest-hwn\Analysis\Noord-Brabant Oost\Outputs\allowed_lowerlying_flooded_dissolved.gpkg | groups=11 | flooded=4


In [ ]:
#On off ramp analysis
from pathlib import Path
import geopandas as gpd
from post_processing_functions import cluster_connected,aggregate_clusters_to_points



for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    network_file = root_dir / "Damages_Filtering_all_columns.gpkg"
    network_gdf = gpd.read_file(network_file)

    ramps_gdf = network_gdf[network_gdf["BST_CODE_NWB"].isin(["AFR", "OPR"])]


    afr_gdf = ramps_gdf[ramps_gdf["BST_CODE_NWB"] == "AFR"].copy()
    opr_gdf = ramps_gdf[ramps_gdf["BST_CODE_NWB"] == "OPR"].copy()

   

    afr_gdf_clustered = cluster_connected(afr_gdf)
    output_gpkg = root_dir / "afr_LineSegments.gpkg"
    afr_gdf_clustered.to_file(output_gpkg, driver="GPKG")

    
    afr_gdf_aggregated = aggregate_clusters_to_points(afr_gdf_clustered, "F_EV1_me", method="max")
    output_gpkg_aggregated_afr = root_dir / "afr_Points.gpkg"
    afr_gdf_aggregated.to_file(output_gpkg_aggregated_afr, driver="GPKG")

   

    opr_gdf_clustered = cluster_connected(opr_gdf)
    
    output_gpkg = root_dir / "opr_LineSegments.gpkg"
    opr_gdf_clustered.to_file(output_gpkg, driver="GPKG")

    opr_gdf_aggregated = aggregate_clusters_to_points(opr_gdf_clustered, "F_EV1_me", method="max")
    output_gpkg_aggregated = root_dir / "opr_Points.gpkg"
    opr_gdf_aggregated.to_file(output_gpkg_aggregated, driver="GPKG")
